In [2]:
# 1. Setup (run first): install Ultralytics and mount Drive
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# 2. Correct vs misclassified test photos for the learning-rate runs (CPU is fine)
import os, glob, shutil, csv, math
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

RUNS = "/content/drive/MyDrive/IAT360CVProject/classification/runs"
DRIVE_DIR = "/content/drive/MyDrive/IAT360CVProject/classification/dataset"
TEST = "/content/dataset/test"
OUT = f"{RUNS}/report_examples"
REF = "lr0_0.005"   # run used for the misclassified-examples figure
RUN_NAMES = ["baseline", "lr0_0.001", "lr0_0.005", "lr0_0.02"]
os.makedirs(OUT, exist_ok=True)

# copy only the test split from Drive (fast) if it is not already local
if not os.path.isdir(TEST):
    os.makedirs("/content/dataset", exist_ok=True)
    shutil.copytree(f"{DRIVE_DIR}/test", TEST)
paths = sorted(glob.glob(f"{TEST}/*/*.*"))
print(len(paths), "test photos")

# classify every test photo with each run's best.pt (streaming keeps RAM low)
results = {}
for run in RUN_NAMES:
    w = f"{RUNS}/{run}/weights/best.pt"
    if not os.path.exists(w):
        print("skipping", run, "- no best.pt found at", w)
        continue
    model = YOLO(w)
    rows = {}
    for r in model.predict(paths, stream=True, verbose=False):
        true = os.path.basename(os.path.dirname(r.path))
        rows[r.path] = (true, r.names[r.probs.top1], float(r.probs.top1conf))
    results[run] = rows
    ok = sum(t == p for t, p, _ in rows.values())
    print(f"{run}: {ok}/{len(rows)} correct")

# one CSV with every photo and every run's prediction
runs_done = list(results)
with open(f"{OUT}/test_predictions_by_run.csv", "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["file", "true"] + [f"{r}_{k}" for r in runs_done for k in ("pred", "conf")])
    for p in paths:
        wr.writerow([os.path.basename(p), os.path.basename(os.path.dirname(p))] +
                    [x for r in runs_done for x in (results[r][p][1], round(results[r][p][2], 3))])

# list the wrong photos per run, and the ones every run gets wrong
for run in runs_done:
    print(f"\n{run} misclassified:")
    for p, (t, pr, c) in results[run].items():
        if t != pr:
            print(f"  {os.path.basename(p)}: true {t} -> predicted {pr} ({c:.2f})")
all_wrong = [p for p in paths if all(results[r][p][0] != results[r][p][1] for r in runs_done)]
print(f"\nMissed by all {len(runs_done)} runs: {len(all_wrong)}")
for p in all_wrong:
    print("  ", os.path.basename(p))

# figure: misclassified photos for the reference run, labelled true vs predicted
ref = REF if REF in results else runs_done[0]
bad = [(p, t, pr, c) for p, (t, pr, c) in results[ref].items() if t != pr]
cols = 4
nrows = max(1, math.ceil(len(bad) / cols))
fig, axes = plt.subplots(nrows, cols, figsize=(cols * 4.2, nrows * 4.8))
axes = axes.ravel()
for ax in axes:
    ax.axis("off")
for ax, (p, t, pr, c) in zip(axes, bad):
    ax.imshow(Image.open(p).convert("RGB"))
    ax.set_title(f"true: {t}\npredicted: {pr} ({c:.2f})", fontsize=11)
plt.tight_layout()
png = f"{OUT}/{ref}_misclassified_examples.png"
plt.savefig(png, dpi=150, bbox_inches="tight")
plt.show()
print("saved", png)

Output hidden; open in https://colab.research.google.com to view.